# ========================================
# 第1部分：项目介绍
# ========================================

# KuzH888-ShoppingAgent

## 项目简介

KuzH888-ShoppingAgent 是一个基于 HelloAgents 的多语言智能购物客服。项目面向商品数量和品类有限的小型综合商城，通过理解用户的使用场景、预算、必要功能和个人偏好，搜索并推荐最适合的商品。最终系统计划采用 Python + FastAPI 后端，以及带有客服弹窗的自定义商城网页。

## 作者信息

- 姓名：
- GitHub：@KuzH888
- 日期：2026-09-09

# ========================================
# 第2部分：环境配置
# ========================================

先安装项目依赖，再加载环境变量和 HelloAgents 组件。真实 API 密钥应保存在项目根目录的 `.env` 中，不要写进 Notebook 或提交到 GitHub。

In [1]:
# 安装项目依赖。首次配置环境时运行一次即可。
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 导入必要的库
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter, ToolResponse
import os
from src.utils.config import (
    list_public_models,
    load_runtime_config,
)

# 加载安全配置；不会打印 API Key
runtime_config = load_runtime_config()

print("运行配置：", runtime_config.public_dict())
print("可选模型：", [model["id"] for model in list_public_models()])

ImportError: cannot import name 'BaseTool' from 'hello_agents.tools' (d:\Hello_Agent\hello-agents\Co-creation-projects\KuzH888-ShoppingAgent\.venv\Lib\site-packages\hello_agents\tools\__init__.py)

# ========================================
# 第3部分：工具定义
# ========================================

KuzMall 的 24 件双语商品已保存在 `data/products.json`，并通过 Pydantic 数据模型验证。下面加载确定性的需求解析、商品搜索、详情查询和比较工具；这些工具在模拟模式下也能运行，不需要 API Key。

In [ ]:
from collections import Counter
from src.utils.catalog import load_catalog

catalog = load_catalog()
category_counts = Counter(product.category.value for product in catalog.products)
print(f"商城：{catalog.store_name}")
print(f"商品总数：{len(catalog.products)}")
print("分类统计：", dict(category_counts))

In [ ]:
from src.tools import CompareProductsTool, ProductDetailsTool, SearchProductsTool

product_search_tool = SearchProductsTool()
product_details_tool = ProductDetailsTool()
compare_products_tool = CompareProductsTool()

print("工具已创建：", [
    product_search_tool.name,
    product_details_tool.name,
    compare_products_tool.name,
])

# ========================================
# 第4部分：智能体构建
# ========================================

创建购物客服编排器。模拟模式使用本地确定性逻辑完成最多两个澄清问题、会话内需求合并和结构化推荐；在线模式则创建 HelloAgents `SimpleAgent` 并注册三个商品工具。

In [ ]:
from src.agents import ShoppingAssistant, create_live_agent

shopping_assistant = ShoppingAssistant()
if runtime_config.simulation_mode:
    agent = None
    print("当前为模拟模式：本地客服编排器已创建，不会连接外部 API。")
else:
    agent = create_live_agent(selected_model=runtime_config.model_id)
    print(f"HelloAgents 购物客服已创建：{runtime_config.model_id}")

# ========================================
# 第5部分：功能演示
# ========================================

模拟模式直接运行本地推荐工具，不产生 API 费用；关闭模拟模式后，同一查询将交给 HelloAgents 智能体，并由智能体调用这些工具。

In [ ]:
# 示例1：中文商品推荐
print("=== 示例1：中文商品推荐 ===")
query = "我需要100澳元以内、适合通勤的耳机，降噪很重要，而且希望轻便。"
if agent is None:
    result = shopping_assistant.chat(query, session_id="notebook-zh")
    print("[模拟模式]\n" + result.message)
else:
    result = agent.run(query)
    print(result)

In [ ]:
# 示例2：英文商品推荐
print("\n=== Example 2: English recommendation ===")
query = (
    "I need lightweight headphones for commuting. My budget is AUD 100, "
    "and noise cancellation is important."
)
if agent is None:
    result = shopping_assistant.chat(query, session_id="notebook-en")
    print("[Simulation mode]\n" + result.message)
else:
    result = agent.run(query)
    print(result)

# ========================================
# 第6部分：性能评估（可选）
# ========================================

当前先运行不依赖 LLM 的双语推荐回归测试；联网阶段再补充回复语言、响应时间和对话质量指标。

- 需求提取准确率：
- 推荐约束满足率：
- 商品信息忠实度：
- 多语言回复成功率：
- 平均响应时间：

In [ ]:
import json
from pathlib import Path
from src.services import RecommendationEngine, parse_customer_need

test_cases = json.loads(Path("data/test_cases.json").read_text(encoding="utf-8"))
engine = RecommendationEngine(catalog)
passed_cases = []
failed_cases = []

for case in test_cases:
    recommendation = engine.recommend(parse_customer_need(case["query"]))
    actual_top = (
        recommendation.recommendations[0].product_id
        if recommendation.recommendations
        else None
    )
    passed = (
        recommendation.status == case["expected_status"]
        and actual_top == case["expected_top_product"]
    )
    (passed_cases if passed else failed_cases).append(case["id"])

evaluation_results = {
    "total": len(test_cases),
    "passed": len(passed_cases),
    "failed": failed_cases,
    "pass_rate": f"{len(passed_cases) / len(test_cases):.1%}",
}
evaluation_results

# ========================================
# 第7部分：总结与展望
# ========================================

## 项目总结

### 实现的功能

- 已建立符合毕业设计模板的 Jupyter Notebook 结构
- 已建立包含 24 件双语虚构商品的 KuzMall 商品数据
- 已完成商品数据模型、唯一性检查和分类数量验证
- 已实现中英文需求解析、硬性过滤和 100 分商品排序
- 已实现商品搜索、详情查询和两至三件商品比较工具
- 已建立 13 个中英文推荐案例和自动化测试
- 已实现最多两个澄清问题、会话内需求合并和同语言结构化回复
- 已实现可选模型的 HelloAgents 在线智能体工厂，模拟阶段不会调用 API
- 已实现前后端分离的 FastAPI JSON 接口和接口自动化测试
- 已实现 Vue 3、TypeScript、Vite 与 Pinia 商城前端和浮动客服窗口

### 遇到的挑战

- 英文短词 `mic` 曾误匹配 `ergonomic`；已改为完整单词边界匹配
- 无精确匹配时不能暗中放宽条件；系统会明确标记最多两个近似备选

### 未来改进方向

- 在最终测试阶段配置 API Key，验证真实 LLM 工具调用
- 增加商品详情、购物车等可选商城能力
- 完成中英文推荐质量评估